<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.4_Praktikum_Thresholding.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.4: Praktikum Thresholding
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengimplementasikan binerisasi global statis (`cv2.threshold`) dengan berbagai tipe flag.
2. Menerapkan algoritma optimasi varians Otsu (`THRESH_OTSU`) pada histogram bimodal citra perkebunan.
3. Menguji ketahanan Adaptive Gaussian Thresholding terhadap citra kanopi dengan gradien bayangan tajuk.
4. Menganalisis pengaruh parameter `blockSize` dan konstanta $C$ terhadap integritas masker segmentasi.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Pembangkitan Citra Sintetis Kanopi dengan Gradien Bayangan Lapangan
Mensimulasikan citra tajuk kelapa sawit yang berada di bawah pengaruh gradien intensitas cahaya akibat sudut datang matahari.

In [ ]:
H, W = 400, 600
# Gradien bayangan tajuk (kiri terang ~ 210, kanan gelap ~ 40)
grad_line = np.linspace(210, 40, W, dtype=np.float32)
synthetic_base = np.tile(grad_line, (H, 1))

# Objek Daun Sawit 1: Di sisi terang (kiri)
cv2.ellipse(synthetic_base, (170, 200), (95, 45), -25, 0, 360, 245, -1)
# Objek Daun Sawit 2: Di sisi bayangan gelap (kanan)
cv2.ellipse(synthetic_base, (430, 200), (95, 45), 25, 0, 360, 115, -1)

canopy_gray = np.clip(synthetic_base, 0, 255).astype(np.uint8)
blurred_canopy = cv2.GaussianBlur(canopy_gray, (5, 5), 0)
print(f'Dimensi Citra Sintetis: {canopy_gray.shape}')


## 2. Pengujian Tiga Metode Thresholding: Global, Otsu, dan Adaptif
Membandingkan segmentasi biner antara ambang batas global tetap, Otsu binarization, dan Adaptive Gaussian.

In [ ]:
# 1. Global Thresholding (T = 127)
_, mask_global = cv2.threshold(blurred_canopy, 127, 255, cv2.THRESH_BINARY)

# 2. Otsu's Binarization Otomatis
otsu_val, mask_otsu = cv2.threshold(blurred_canopy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. Adaptive Gaussian Thresholding (BlockSize=31, C=5)
mask_adapt = cv2.adaptiveThreshold(
    blurred_canopy, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 5
)

print(f'Nilai Ambang Batas Optimal Otsu: T* = {otsu_val:.1f}')
print(f'Piksel Putih Tersegmentasi (Global) : {cv2.countNonZero(mask_global)}')
print(f'Piksel Putih Tersegmentasi (Otsu)   : {cv2.countNonZero(mask_otsu)}')
print(f'Piksel Putih Tersegmentasi (Adaptif): {cv2.countNonZero(mask_adapt)}')


## 3. Visualisasi Komparasi Kinerja Penanganan Bayangan Tajuk
Menampilkan citra asli dan perbandingan masker biner dari ketiga metode penambangan.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(canopy_gray, cmap='gray')
axes[0].set_title('1. Citra Asli (Gradien Bayangan)', fontsize=10, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(mask_global, cmap='gray')
axes[1].set_title('2. Global (T=127): Gagal di Bayangan', fontsize=10, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(mask_otsu, cmap='gray')
axes[2].set_title(f'3. Otsu (T*={otsu_val:.0f}): Gagal Separasi', fontsize=10, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(mask_adapt, cmap='gray')
axes[3].set_title('4. Adaptive: Kedua Daun Terisolasi!', fontsize=10, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.savefig('praktikum_thresholding_komparasi_11_4.png', dpi=150)
plt.show()
